# Build sum-of-products KEO for formaldehyde ($\mathrm{H_2CO}$) using Z-matrix coordinates

First, we compare the sum-of-products form with the numerical approach using small 1D grids to avoid generating large multidimensional representations that are needed only for comparison.

Then, we evaluate the sum-of-products form using larger 1D grids and store the results in a file.

In [1]:
import jax
import numpy as np
from scipy import constants

from keosop.gmat import Gmat
from keosop.hdf5 import load_gmat_hdf5, save_gmat_hdf5
from keosop.numerical.keo import batch_Gmat, com
from keosop.numerical.zmat import internal_to_cartesian_jax
from keosop.zmat import cartesian_to_internal, internal_to_cartesian

jax.config.update("jax_enable_x64", True)

# converts G-matrix to cm^-1 units
G_to_invcm = (
    constants.value("Planck constant")
    * constants.value("Avogadro constant")
    * 1e16
    / (4.0 * np.pi**2 * constants.value("speed of light in vacuum"))
    * 1e5
)

Define Z-matrix coordinates for $\mathrm{H_2CO}$

In [2]:
masses = np.array([12.0, 15.99491463, 1.00782505, 1.00782505])  # masses of C, O, H, H

zmat_connections = [
    (0, 0, 0, 0),  # C
    (1, 0, 0, 0),  # O  - C
    (1, 2, 0, 0),  # H1 - C - O
    (1, 2, 3, 0),  # H2 - C - O - H1
]

Define one-dimensional grids for the internal coordinate representation and construct the $G$-matrix in sum-of-products form

In [3]:
rad = np.pi / 180

# define one-dimensional grids for each coordinate
grids = [
    np.linspace(0.8, 2.0, 5),  # r_{CO}
    np.linspace(0.8, 2.0, 5),  # r_{CH1}
    np.linspace(0.8, 2.0, 5),  # r_{CH2}
    np.linspace(30 * rad, 160 * rad, 5),  # alpha_{H1CO}
    np.linspace(30 * rad, 160 * rad, 5),  # alpha_{H2CO}
    np.linspace(30 * rad, 330 * rad, 5),  # tau
]

ncoo = len(grids)

# compute SOP for G-matrix
gmat = Gmat(
    grids,
    masses,
    lambda q: internal_to_cartesian(zmat_connections, q),
    lambda xyz: cartesian_to_internal(zmat_connections, xyz),
)

gvib_sop, grovib_sop, grot_sop = gmat.run(return_rot=True)

print("Number of terms for vibrational elements of G-matrix\n i  j no.terms")
for i in range(ncoo):
    for j in range(i, ncoo):
        print(f"{i:2d} {j:2d} {gvib_sop[i, j].nterms:6d}")

print("\nNumber of terms for rovibrational elements of G-matrix\n i  j no.terms")
for i in range(3):
    for j in range(ncoo):
        print(f"{i:2d} {j:2d} {grovib_sop[i, j].nterms:6d}")

print("\nNumber of terms for rotational elements of G-matrix\n i  j no.terms")
for i in range(3):
    for j in range(i, 3):
        print(f"{i:2d} {j:2d} {grot_sop[i, j].nterms:6d}")

Number of terms for vibrational elements of G-matrix
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 0  3      2
 0  4      2
 0  5      1
 1  1      1
 1  2      2
 1  3      1
 1  4      4
 1  5      2
 2  2      1
 2  3      4
 2  4      1
 2  5      2
 3  3      4
 3  4      8
 3  5      4
 4  4      4
 4  5      4
 5  5     10

Number of terms for rovibrational elements of G-matrix
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 0  3      1
 0  4      2
 0  5      4
 1  0      1
 1  1      1
 1  2      1
 1  3      2
 1  4      2
 1  5      2
 2  0      1
 2  1      1
 2  2      2
 2  3      1
 2  4      4
 2  5      7

Number of terms for rotational elements of G-matrix
 i  j no.terms
 0  0      1
 0  1      1
 0  2      2
 1  1      1
 1  2      1
 2  2      3


Evaluate the $G$-matrix numerically on the multidimensional grid formed by the Cartesian product of the one-dimensional grids used to construct the sum-of-products representation

In [4]:
# build direct-product grid
product_grid = np.stack(np.meshgrid(*grids, indexing="ij"), axis=-1)
product_grid = product_grid.reshape(-1, len(grids))

# map each multidimensional grid point to its 1D grid indices
grid_indices = np.indices([len(grid) for grid in grids]).reshape(len(grids), -1).T

# compute G-matrix on grid
gmat = batch_Gmat(
    product_grid,
    masses,
    # lambda q: com(masses)(internal_to_cartesian_jax)(zmat_connections, q),
    lambda q: internal_to_cartesian_jax(zmat_connections, q),
)

gvib_num = gmat[:, :ncoo, :ncoo]
grovib_num = gmat[:, ncoo : ncoo + 3, :ncoo]
grot_num = gmat[:, ncoo : ncoo + 3, ncoo : ncoo + 3]

Calculate point-wise differences between the sum-of-products and numerical $G$-matrix

In [5]:
# evaluate one SOP term
def eval_sop_term(sop, iterm, inds):
    value = 1.0
    for k in range(sop.ncoo):
        value *= sop.terms[iterm][k][inds[k]]
    return value

# evaluate a complete SumProd object at one direct-product grid point
def eval_sop(sop, inds):
    return sum(
        eval_sop_term(sop, iterm, inds)
        for iterm in range(sop.nterms)
    )

In [6]:
# number of multidimensional grid points
npoints = product_grid.shape[0]

gvib_sop_eval = np.zeros((npoints, ncoo, ncoo), dtype=np.float64)
grot_sop_eval = np.zeros((npoints, 3, 3), dtype=np.float64)
grovib_sop_eval = np.zeros((npoints, 3, ncoo), dtype=np.float64)

# Gvib
for ipoint, inds in enumerate(grid_indices):
    for i in range(ncoo):
        for j in range(ncoo):
            gvib_sop_eval[ipoint, i, j] = eval_sop(gvib_sop[i, j], inds) * G_to_invcm


# Grot
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(3):
            grot_sop_eval[ipoint, i, j] = eval_sop(grot_sop[i, j], inds) * G_to_invcm


# Grovib
for ipoint, inds in enumerate(grid_indices):
    for i in range(3):
        for j in range(ncoo):
            grovib_sop_eval[ipoint, i, j] = (
                eval_sop(grovib_sop[i, j], inds) * G_to_invcm
            )

In [7]:
# differences

diff_vib = gvib_sop_eval - gvib_num
diff_rovib = grovib_sop_eval - grovib_num
diff_rot = grot_sop_eval - grot_num

abs_diff_vib = np.abs(diff_vib)
abs_diff_rovib = np.abs(diff_rovib)
abs_diff_rot = np.abs(diff_rot)

print("--- Gvib ---")
print("Maximum absolute difference:", np.max(abs_diff_vib))
print("Mean absolute difference:", np.mean(abs_diff_vib))
print("RMS difference:", np.sqrt(np.mean(diff_vib**2)))

print("\n--- Grovib ---")
print("Maximum absolute difference:", np.max(abs_diff_rovib))
print("Mean absolute difference:", np.mean(abs_diff_rovib))
print("RMS difference:", np.sqrt(np.mean(diff_rovib**2)))

print("\n--- Grot ---")
print("Maximum absolute difference:", np.max(abs_diff_rot))
print("Mean absolute difference:", np.mean(abs_diff_rot))
print("RMS difference:", np.sqrt(np.mean(diff_rot**2)))

--- Gvib ---
Maximum absolute difference: 8.071765478234738e-12
Mean absolute difference: 5.872474515272856e-15
RMS difference: 5.171311853618632e-14

--- Grovib ---
Maximum absolute difference: 8.29913915367797e-12
Mean absolute difference: 5.750973685211991e-15
RMS difference: 6.808154612979074e-14

--- Grot ---
Maximum absolute difference: 8.071765478234738e-12
Mean absolute difference: 1.032369929897528e-14
RMS difference: 9.565938814738342e-14


In [8]:
print("--- Gvib element-wise errors ---")

for i in range(ncoo):
    for j in range(i, ncoo):
        d = diff_vib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(f"Gvib[{i},{j}]  " f"max abs = {max_abs:.3e}   " f"RMS = {rms:.3e}   ")

print("\n--- Grovib element-wise errors ---")

for i in range(3):
    for j in range(ncoo):
        d = diff_rovib[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(f"Grovib[{i},{j}]  " f"max abs = {max_abs:.3e}   " f"RMS = {rms:.3e}   ")

print("\n--- Grot element-wise errors ---")

for i in range(3):
    for j in range(i, 3):
        d = diff_rot[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(f"Grot[{i},{j}]  " f"max abs = {max_abs:.3e}   " f"RMS = {rms:.3e}   ")

--- Gvib element-wise errors ---
Gvib[0,0]  max abs = 8.882e-16   RMS = 6.847e-16   
Gvib[0,1]  max abs = 2.220e-15   RMS = 6.072e-16   
Gvib[0,2]  max abs = 2.220e-15   RMS = 6.476e-16   
Gvib[0,3]  max abs = 5.995e-15   RMS = 5.627e-16   
Gvib[0,4]  max abs = 2.665e-15   RMS = 5.926e-16   
Gvib[0,5]  max abs = 1.091e-14   RMS = 5.061e-16   
Gvib[1,1]  max abs = 1.421e-14   RMS = 6.382e-15   
Gvib[1,2]  max abs = 5.329e-15   RMS = 8.110e-16   
Gvib[1,3]  max abs = 1.121e-14   RMS = 1.890e-15   
Gvib[1,4]  max abs = 1.066e-14   RMS = 1.147e-15   
Gvib[1,5]  max abs = 2.842e-14   RMS = 2.599e-15   
Gvib[2,2]  max abs = 2.842e-14   RMS = 5.698e-15   
Gvib[2,3]  max abs = 2.498e-14   RMS = 1.306e-15   
Gvib[2,4]  max abs = 1.510e-14   RMS = 2.554e-15   
Gvib[2,5]  max abs = 1.563e-13   RMS = 7.385e-15   
Gvib[3,3]  max abs = 1.847e-13   RMS = 2.327e-14   
Gvib[3,4]  max abs = 7.816e-14   RMS = 5.091e-15   
Gvib[3,5]  max abs = 6.479e-13   RMS = 2.267e-14   
Gvib[4,4]  max abs = 1.421e-13 

In [9]:
print("--- Gvib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_vib), abs_diff_vib.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(product_grid[ipoint]))
print("element    :", (i, j))
print("SOP        :", gvib_sop_eval[ipoint, i, j])
print("reference  :", gvib_num[ipoint, i, j])
print("difference :", diff_vib[ipoint, i, j])

print("\n--- Grovib largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rovib), abs_diff_rovib.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(product_grid[ipoint]))
print("element    :", (i, j))
print("SOP        :", grovib_sop_eval[ipoint, i, j])
print("reference  :", grovib_num[ipoint, i, j])
print("difference :", diff_rovib[ipoint, i, j])

print("\n--- Grot largest discrepancy ---")
idx = np.unravel_index(np.argmax(abs_diff_rot), abs_diff_rot.shape)
ipoint, i, j = idx

print("grid point :", ipoint)
print("coordinates:", np.asarray(product_grid[ipoint]))
print("element    :", (i, j))
print("SOP        :", grot_sop_eval[ipoint, i, j])
print("reference  :", grot_num[ipoint, i, j])
print("difference :", diff_rot[ipoint, i, j])

--- Gvib largest discrepancy ---
grid point : 612
coordinates: [0.8        0.8        2.         2.7925268  1.65806279 3.14159265]
element    : (np.int64(5), np.int64(5))
SOP        : 648.3415172772853
reference  : 648.3415172772934
difference : -8.071765478234738e-12

--- Grovib largest discrepancy ---
grid point : 612
coordinates: [0.8        0.8        2.         2.7925268  1.65806279 3.14159265]
element    : (np.int64(2), np.int64(5))
SOP        : -625.8723937675906
reference  : -625.8723937675989
difference : 8.29913915367797e-12

--- Grot largest discrepancy ---
grid point : 612
coordinates: [0.8        0.8        2.         2.7925268  1.65806279 3.14159265]
element    : (np.int64(2), np.int64(2))
SOP        : 612.9056805240167
reference  : 612.9056805240248
difference : -8.071765478234738e-12


Now, generate the sum-of-products KEO using larger 1D grids and store the results in an HDF5 file

In [10]:
rad = np.pi / 180

# define one-dimensional grids for each coordinate
grids = [
    np.linspace(0.8, 2.6, 5000),  # r_{CO}
    np.linspace(0.8, 2.6, 5000),  # r_{CH1}
    np.linspace(0.8, 2.6, 5000),  # r_{CH2}
    np.linspace(30 * rad, 160 * rad, 5000),  # alpha_{H1CO}
    np.linspace(30 * rad, 160 * rad, 5000),  # alpha_{H2CO}
    np.linspace(30 * rad, 330 * rad, 5000),  # tau
]

ncoo = len(grids)

# compute SOP for G-matrix
gmat = Gmat(
    grids,
    masses,
    lambda q: internal_to_cartesian(zmat_connections, q),
    lambda xyz: cartesian_to_internal(zmat_connections, xyz),
)

gvib_sop, grovib_sop, grot_sop = gmat.run(return_rot=True)

# store into file
filename = "keo_h2co_zmat.hdf5"
save_gmat_hdf5(
    filename,
    grids,
    gvib_sop,
    grovib_sop,
    grot_sop,
    coo_names=("r_CO", "r_CH1", "r_CH2", "alpha_H1CO", "alpha_H2CO", "tau_H1COH2"),
    coo_units=("angs", "angs", "angs", "rad", "rad", "rad"),
)

Test-load the $G$-matrices from file and print the number of sum-of-products terms for each element

In [11]:
grids, gvib, grovib, grot = load_gmat_hdf5(filename)

print("Number of terms for vibrational elements of G-matrix\n i  j no.terms")
for i in range(ncoo):
    for j in range(i, ncoo):
        print(f"{i:2d} {j:2d} {gvib[i, j].nterms:6d}")

print("\nNumber of terms for rovibrational elements of G-matrix\n i  j no.terms")
for i in range(3):
    for j in range(ncoo):
        print(f"{i:2d} {j:2d} {grovib[i, j].nterms:6d}")

print("\nNumber of terms for rotational elements of G-matrix\n i  j no.terms")
for i in range(3):
    for j in range(i, 3):
        print(f"{i:2d} {j:2d} {grot[i, j].nterms:6d}")

Number of terms for vibrational elements of G-matrix
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 0  3      2
 0  4      2
 0  5      1
 1  1      1
 1  2      2
 1  3      2
 1  4      4
 1  5      2
 2  2      1
 2  3      4
 2  4      2
 2  5      2
 3  3      6
 3  4      8
 3  5      4
 4  4      6
 4  5      4
 5  5     10

Number of terms for rovibrational elements of G-matrix
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 0  3      1
 0  4      2
 0  5      4
 1  0      1
 1  1      1
 1  2      1
 1  3      2
 1  4      2
 1  5      2
 2  0      1
 2  1      1
 2  2      2
 2  3      1
 2  4      4
 2  5      7

Number of terms for rotational elements of G-matrix
 i  j no.terms
 0  0      1
 0  1      1
 0  2      2
 1  1      1
 1  2      1
 2  2      3
